# Using Online Customer Rating to Examine Restaurant Health Inspection Results

**Team members:** Raffaella Alpaca, Helen Ayalew, Angel Pavu

**Repository:** https://github.com/hwayalew/ds3001_team1_project

## Setup

Use relative paths and keep the notebook runnable from beginning to end.

In [1]:
from pathlib import Path

# Download our team's GitHub repository into Colab
!git clone https://github.com/hwayalew/ds3001_team1_project.git

# Set the path to our data folder
DATA_DIR = Path("ds3001_team1_project/data")

# Check that the folder exists
assert DATA_DIR.is_dir(), "The data directory is missing."
print("Data folder found!")


Cloning into 'ds3001_team1_project'...
remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 46 (delta 9), reused 3 (delta 3), pack-reused 29 (from 1)
Receiving objects: 100% (46/46), 2.16 MiB | 8.69 MiB/s, done.
Resolving deltas: 100% (18/18), done.
Data folder found!


In [2]:
# Show all files inside our data folder
for file in DATA_DIR.iterdir():
    print(file.name)


food_inspections_all.xlsx
fairfax_falls_church_google_restaurants.csv
README.md


## Load data

In [3]:
# Load data from DATA_DIR.

In [4]:
import pandas as pd
# Load the food inspection data from Excel
inspections = pd.read_excel(
    DATA_DIR / "food_inspections_all.xlsx",
    sheet_name="Inspections"
)
# Load the establishment data from the same Excel file
establishments = pd.read_excel(
    DATA_DIR / "food_inspections_all.xlsx",
    sheet_name="Establishments"
)

# Load the Google restaurant ratings from CSV
google = pd.read_csv(
    DATA_DIR / "fairfax_falls_church_google_restaurants.csv"
)

# Check that everything loaded correctly
print("Inspections:", inspections.shape)
print("Establishments:", establishments.shape)
print("Google Ratings:", google.shape)

Inspections: (13271, 13)
Establishments: (3840, 9)
Google Ratings: (876, 12)


## Prepare data

In [5]:
# Clean data and create features.

In [10]:
# Prepare data for analysis
# Make copies so we keep the original datasets unchanged
inspections_clean = inspections.copy()
establishments_clean = establishments.copy()
google_clean = google.copy()

#check missing values in each dataset
print("Missing values in inspections:")
print(inspections_clean.isnull().sum())

print("\nMissing values in establishments:")
print(establishments_clean.isnull().sum())

print("\nMissing values in Google ratings:")
print(google_clean.isnull().sum())

#check the column names before cleaning and merging
print("\nInspection columns:")
print(inspections_clean.columns.tolist())

print("\nEstablishment columns:")
print(establishments_clean.columns.tolist())

print("\nGoogle columns:")
print(google_clean.columns.tolist())


Missing values in inspections:
INSPECTIONID           0
RECORDID              11
ESTABLISHMENT_NAME     1
ESTABLISHMENT_TYPE    12
ADDRESS               12
CITY                  12
STATE                 12
ZIP                   12
JURISDICTION          12
FOOD_PLACE            12
INSPECTIONDATE         0
INSPECTIONRESULT       0
WEBSITE_LINK           0
dtype: int64

Missing values in establishments:
RECORDID              0
ESTABLISHMENT_NAME    0
ESTABLISHMENT_TYPE    0
ADDRESS               0
CITY                  0
STATE                 0
ZIP                   0
JURISDICTION          0
FOOD_PLACE            0
dtype: int64

Missing values in Google ratings:
gmap_id              0
name                 0
address              0
city                 0
postal_code          0
categories           0
avg_rating           0
num_of_reviews       0
price              238
latitude             0
longitude            0
google_maps_url      0
dtype: int64

Inspection columns:
['INSPECTIONID', 'RECO

In [11]:
# Remove inspection records missing important information
#need these fields to identify and match restaurants!!!!
inspections_clean = inspections_clean.dropna(
    subset=["RECORDID", "ESTABLISHMENT_NAME", "ADDRESS", "CITY"]
)

# Google price is optional, so only remove rows missing essential information for matching and analysis
google_clean = google_clean.dropna(
    subset=["name", "address", "city", "avg_rating"]
)
#how many rows remain
print("Inspections after cleaning:", inspections_clean.shape)
print("Establishments after cleaning:", establishments_clean.shape)
print("Google ratings after cleaning:", google_clean.shape)


Inspections after cleaning: (13259, 13)
Establishments after cleaning: (3840, 9)
Google ratings after cleaning: (876, 12)


We remove 12 inspection rows missing key identifying information, provided the 11 missing RECORDID values are among those same 12 records.

In [12]:
#keep only Fairfax and Falls Church restaurants
cities = ["FAIRFAX", "FALLS CHURCH"]

#filter inspection data
inspections_clean = inspections_clean[
    inspections_clean["CITY"].str.upper().isin(cities)
].copy()

#filter establishment data
establishments_clean = establishments_clean[
    establishments_clean["CITY"].str.upper().isin(cities)
].copy()

#filter Google ratings
google_clean = google_clean[
    google_clean["city"].str.upper().isin(cities)
].copy()

#check the remaining rows
print("Inspections:", inspections_clean.shape)
print("Establishments:", establishments_clean.shape)
print("Google ratings:", google_clean.shape)


Inspections: (3863, 13)
Establishments: (1124, 9)
Google ratings: (876, 12)


In [14]:
#count each type of inspection result
print(inspections_clean["INSPECTIONRESULT"].value_counts())

INSPECTIONRESULT
Passed                  2958
Partial Pass             871
Failed - Self Closed      33
Failed                     1
Name: count, dtype: int64


In [15]:
#Check whether establishment IDs are unique
print("Duplicate establishment IDs:")
print(establishments_clean["RECORDID"].duplicated().sum())

# Check how many unique establishments have inspections
print("\nUnique establishments in inspections:")
print(inspections_clean["RECORDID"].nunique())

# Check whether inspection records have matching establishments
matches = inspections_clean["RECORDID"].isin(
    establishments_clean["RECORDID"]
)
print("\nInspections with matching establishments:")
print(matches.sum())

print("\nInspections without matching establishments:")
print((~matches).sum())


Duplicate establishment IDs:
0

Unique establishments in inspections:
1124

Inspections with matching establishments:
3863

Inspections without matching establishments:
0


In [16]:
#merge inspection records with establishment information
# Keep RECORDID from establishments for matching
establishment_ids = establishments_clean[["RECORDID"]]

# Combine the datasets using RECORDID
merged_inspections = pd.merge(
    inspections_clean,
    establishment_ids,
    on="RECORDID",
    how="inner",
    validate="many_to_one"
)

#check the merged dataset
print("Merged data shape:", merged_inspections.shape)
display(merged_inspections.head())

Merged data shape: (3863, 13)


,INSPECTIONID,RECORDID,ESTABLISHMENT_NAME,ESTABLISHMENT_TYPE,ADDRESS,CITY,STATE,ZIP,JURISDICTION,FOOD_PLACE,INSPECTIONDATE,INSPECTIONRESULT,WEBSITE_LINK
0,3534783,HFOOD-2023-00135,"""Robeks Fresh Juices & Smoothies""",Fast Food Restaurant,1063 BROAD ST,FALLS CHURCH,VA,22046,CITY OF FALLS CHURCH,Yes,2024-11-08,Passed,https://plus.fairfaxcounty.gov/CitizenAccess/u...
1,3677656,HFOOD-2023-00135,"""Robeks Fresh Juices & Smoothies""",Fast Food Restaurant,1063 BROAD ST,FALLS CHURCH,VA,22046,CITY OF FALLS CHURCH,Yes,2025-04-15,Passed,https://plus.fairfaxcounty.gov/CitizenAccess/u...
2,3877055,HFOOD-2023-00135,"""Robeks Fresh Juices & Smoothies""",Fast Food Restaurant,1063 BROAD ST,FALLS CHURCH,VA,22046,CITY OF FALLS CHURCH,Yes,2025-10-16,Passed,https://plus.fairfaxcounty.gov/CitizenAccess/u...
3,3529095,HFOOD-2021-00045,&PIZZA,Fast Food Restaurant,6110C ARLINGTON BLVD,FALLS CHURCH,VA,22044,FAIRFAX COUNTY,Yes,2024-11-01,Passed,https://plus.fairfaxcounty.gov/CitizenAccess/u...
4,3569188,HFOOD-2021-00310,&PIZZA,Fast Food Restaurant,11824KL FAIR OAKS SHOPPING CTR,FAIRFAX,VA,22033,FAIRFAX COUNTY,Yes,2024-12-16,Partial Pass,https://plus.fairfaxcounty.gov/CitizenAccess/u...


In [17]:
# Standardize restaurant names for matching
# Make copies to keep the original data unchanged
merged_inspections = merged_inspections.copy()
google_clean = google_clean.copy()

# Clean Fairfax restaurant names
merged_inspections["name_clean"] = (
    merged_inspections["ESTABLISHMENT_NAME"]
    .str.lower()
    .str.replace(r"[^\w\s]", "", regex=True)
    .str.strip()
)
# Clean Google restaurant names
google_clean["name_clean"] = (
    google_clean["name"]
    .str.lower()
    .str.replace(r"[^\w\s]", "", regex=True)
    .str.strip()
)
# Display examples
print("Fairfax restaurant names:")
display(
    merged_inspections[
        ["ESTABLISHMENT_NAME", "name_clean"]
    ].head()
)

print("Google restaurant names:")
display(
    google_clean[
        ["name", "name_clean"]
    ].head()
)

Fairfax restaurant names:


,ESTABLISHMENT_NAME,name_clean
0,"""Robeks Fresh Juices & Smoothies""",robeks fresh juices smoothies
1,"""Robeks Fresh Juices & Smoothies""",robeks fresh juices smoothies
2,"""Robeks Fresh Juices & Smoothies""",robeks fresh juices smoothies
3,&PIZZA,pizza
4,&PIZZA,pizza


Google restaurant names:


,name,name_clean
0,21 Great American Bistro,21 great american bistro
1,29 Diner,29 diner
2,703 Bar + Kitchen,703 bar kitchen
3,99C Hot Pot,99c hot pot
4,AABEE Persian & Mediterranean Restaurant,aabee persian mediterranean restaurant


In [18]:
# Standardize restaurant addresses for matching

# Clean Fairfax addresses
merged_inspections["address_clean"] = (
    merged_inspections["ADDRESS"]
    .str.lower()
    .str.replace(r"[^\w\s]", " ", regex=True)
    .str.replace(r"\bstreet\b", "st", regex=True)
    .str.replace(r"\broad\b", "rd", regex=True)
    .str.replace(r"\bavenue\b", "ave", regex=True)
    .str.replace(r"\bboulevard\b", "blvd", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Clean Google addresses
google_clean["address_clean"] = (
    google_clean["address"]
    .str.lower()
    .str.replace(r"[^\w\s]", " ", regex=True)
    .str.replace(r"\bstreet\b", "st", regex=True)
    .str.replace(r"\broad\b", "rd", regex=True)
    .str.replace(r"\bavenue\b", "ave", regex=True)
    .str.replace(r"\bboulevard\b", "blvd", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Preview the cleaned addresses
print("Fairfax addresses:")
display(merged_inspections[["ADDRESS", "address_clean"]].head())

print("Google addresses:")
display(google_clean[["address", "address_clean"]].head())

Fairfax addresses:


,ADDRESS,address_clean
0,1063 BROAD ST,1063 broad st
1,1063 BROAD ST,1063 broad st
2,1063 BROAD ST,1063 broad st
3,6110C ARLINGTON BLVD,6110c arlington blvd
4,11824KL FAIR OAKS SHOPPING CTR,11824kl fair oaks shopping ctr


Google addresses:


,address,address_clean
0,"21 Great American Bistro, 10427 North St #102,...",21 great american bistro 10427 north st 102 fa...
1,"29 Diner, 10536 Fairfax Blvd, Fairfax, VA 22030",29 diner 10536 fairfax blvd fairfax va 22030
2,"703 Bar + Kitchen, 11787 Lee Jackson Memorial ...",703 bar kitchen 11787 lee jackson memorial hwy...
3,"99C Hot Pot, 9992 Main St, Fairfax, VA 22031",99c hot pot 9992 main st fairfax va 22031
4,"AABEE Persian & Mediterranean Restaurant, Fair...",aabee persian mediterranean restaurant fairfax...


In [19]:
#extracts the street address from Google records

# Split the address into parts using commas
google_clean["street_address"] = (
    google_clean["address"]
    .str.split(",")
    .str[1]
    .str.strip()
)

# Clean the extracted street addresses
google_clean["street_address"] = (
    google_clean["street_address"]
    .str.lower()
    .str.replace(r"[^\w\s]", " ", regex=True)
    .str.replace(r"\bstreet\b", "st", regex=True)
    .str.replace(r"\broad\b", "rd", regex=True)
    .str.replace(r"\bavenue\b", "ave", regex=True)
    .str.replace(r"\bboulevard\b", "blvd", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Compare the original and extracted addresses
display(
    google_clean[
        ["name", "address", "street_address"]
    ].head(10)
)

# Check how many addresses could not be extracted
print("Missing street addresses:",
      google_clean["street_address"].isnull().sum())

,name,address,street_address
0,21 Great American Bistro,"21 Great American Bistro, 10427 North St #102,...",10427 north st 102
1,29 Diner,"29 Diner, 10536 Fairfax Blvd, Fairfax, VA 22030",10536 fairfax blvd
2,703 Bar + Kitchen,"703 Bar + Kitchen, 11787 Lee Jackson Memorial ...",11787 lee jackson memorial hwy
3,99C Hot Pot,"99C Hot Pot, 9992 Main St, Fairfax, VA 22031",9992 main st
4,AABEE Persian & Mediterranean Restaurant,"AABEE Persian & Mediterranean Restaurant, Fair...",fairfax blvd farr ave
5,ABC Canteen,"ABC Canteen, 9528 Lee Hwy, Fairfax, VA 22031",9528 lee hwy
6,Al-Maza Lebanese Mediterranean,"Al-Maza Lebanese Mediterranean, 10427 North St...",10427 north st
7,"Alta Strada, Mosaic","Alta Strada, Mosaic, 2911 District Ave #150, F...",mosaic
8,Angelico Pizza,"Angelico Pizza, 11391 Government Center Pkwy, ...",11391 government center pkwy
9,Angelika Film Center & Café at Mosaic,"Angelika Film Center & Café at Mosaic, 2911 Di...",2911 district ave suite 200


Missing street addresses: 0


not every extracted address is correct^^^

## Explore data

In [6]:
# Summarize and visualize the prepared data.

## Build models

In [7]:
# Train baseline and candidate models.

## Evaluate models

In [8]:
# Compare models using the chosen evaluation measures.

## Final results

In [9]:
# Generate the final tables and figures.

## AI usage disclosure

- **AI tool/model:** [Name the tool and model, or state that no generative AI was used.]
- **Used for:** [Briefly identify the tasks.]
- **How we verified the work:** [Briefly identify the checks.]
- **Prompt(s) used:** [Paste the prompts used.]